In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
REPORTS_DIR = ROOT / "reports"


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## 1. MBLEDHJA E TË DHËNAVE

In [3]:
section("1. MBLEDHJA E TË DHËNAVE")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rreshta: {df_raw.shape[0]:,}  |  Kolona: {df_raw.shape[1]}")
print(df_raw.head(3).T)

df_raw.info(memory_usage="deep")


1. MBLEDHJA E TË DHËNAVE
Rreshta: 46,831  |  Kolona: 42
                                                                               0                                                  1                                                  2
gbifID                                                                6500474327                                         6500462703                                         6500450749
scientificName                                    Mantis religiosa (Linne, 1758)                  Issoria lathonia (Linnaeus, 1758)                  Podarcis muralis (Laurenti, 1768)
species                                                         Mantis religiosa                                   Issoria lathonia                                   Podarcis muralis
vernacularName                                                    Praying Mantis                          Queen of Spain Fritillary                                 Common Wall Lizard
kingdom                     

## Madhësia e skedarëve

In [4]:
section("Madhësia e skedarëve")
GITHUB_LIMIT_MB = 100


def file_size_mb(path):
    return path.stat().st_size / 1024**2


print(f"Dataseti origjinal: {file_size_mb(DATA_PATH):.1f} MB")
print(f"Në memorie (pandas): {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

drop_planned = ["iasAppliesFrom", "establishmentMeans", "identifiedBy", "iucnRedListStatus",
                "taxonKey", "catalogNumber", "collectionCode", "issue"]
cleaned_mb = len(df_raw.drop(columns=drop_planned).to_csv(index=False).encode("utf-8")) / 1024**2
sample_mb = len(df_raw.sample(frac=0.1, random_state=42).to_csv(index=False).encode("utf-8")) / 1024**2
print(f"Versioni i pastruar (vlerësim): {cleaned_mb:.1f} MB")
print(f"Mostra 10% (vlerësim): {sample_mb:.1f} MB")
print(f"Kufiri i GitHub: {GITHUB_LIMIT_MB} MB")


Madhësia e skedarëve
Dataseti origjinal: 22.0 MB
Në memorie (pandas): 31.9 MB
Versioni i pastruar (vlerësim): 17.5 MB
Mostra 10% (vlerësim): 2.3 MB
Kufiri i GitHub: 100 MB


## 2. DEFINIMI I TIPEVE TË TË DHËNAVE

In [5]:
section("2. DEFINIMI I TIPEVE TË TË DHËNAVE")
df = df_raw.copy()

int_cols = ["year", "month", "day", "individualCount", "taxonKey", "speciesKey"]
for c in int_cols:
    df[c] = df[c].astype("Int64")

cat_cols = [
    "kingdom", "phylum", "class", "order", "family", "taxonRank",
    "iucnRedListStatus", "municipality", "district", "protectedArea",
    "protectedAreaDesignation", "basisOfRecord", "establishmentMeans",
    "institutionCode", "collectionCode", "datasetKey", "license", "iasAppliesFrom",
]
for c in cat_cols:
    df[c] = df[c].astype("category")

iucn_order = ["LC", "NT", "VU", "EN", "CR", "EW", "EX"]
df["iucnRedListCategory"] = pd.Categorical(
    df["iucnRedListCategory"].replace({"NE": np.nan, "DD": np.nan}),
    categories=iucn_order, ordered=True,
)

mem_before = df_raw.memory_usage(deep=True).sum() / 1024**2
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria: {mem_before:.1f} MB -> {mem_after:.1f} MB")

type_summary = pd.DataFrame({
    "tipi_origjinal": df_raw.dtypes.astype(str),
    "tipi_i_ri": df.dtypes.astype(str),
    "vlera_unike": df_raw.nunique(),
})
print(type_summary[type_summary.tipi_origjinal != type_summary.tipi_i_ri])


2. DEFINIMI I TIPEVE TË TË DHËNAVE
Memoria: 31.9 MB -> 17.5 MB
                         tipi_origjinal tipi_i_ri  vlera_unike
kingdom                             str  category            7
phylum                              str  category           25
class                               str  category           68
order                               str  category          226
family                              str  category          683
taxonRank                           str  category           11
taxonKey                        float64     Int64         4647
speciesKey                      float64     Int64         4097
iucnRedListCategory                 str  category            8
iucnRedListStatus                   str  category            8
year                            float64     Int64          102
month                           float64     Int64           12
day                             float64     Int64           31
municipality                        str  category     

## 3.1 VLERAT E ZBRAZËTA

In [6]:
section("3.1 VLERAT E ZBRAZËTA")
missing = pd.DataFrame({
    "mungojne": df_raw.isna().sum(),
    "perqindja": (df_raw.isna().mean() * 100).round(2),
}).sort_values("perqindja", ascending=False)
print(missing[missing.mungojne > 0])

print("\nRreshta pa asnjë vlerë që mungon:", df_raw.dropna().shape[0])
print("Kolona komplet:", missing.index[missing.mungojne == 0].tolist())

print(df_raw["iucnRedListCategory"].value_counts(dropna=False))


3.1 VLERAT E ZBRAZËTA
                               mungojne  perqindja
iasAppliesFrom                    46807      99.95
establishmentMeans                46591      99.49
identifiedBy                      39096      83.48
elevation                         37011      79.03
coordinateUncertaintyInMeters     32004      68.34
protectedArea                     25762      55.01
protectedAreaDesignation          25762      55.01
individualCount                   17697      37.79
locality                          15251      32.57
collectionCode                    13469      28.76
catalogNumber                     12559      26.82
vernacularName                     3873       8.27
recordedBy                         3853       8.23
institutionCode                    3711       7.92
day                                3219       6.87
month                              2867       6.12
year                               2268       4.84
eventDate                          2190       4.68
issue   

## 3.2 DUPLIKATET

In [7]:
section("3.2 DUPLIKATET")
print("Rreshta identikë:", df_raw.duplicated().sum())
print("gbifID të përsëritura:", df_raw["gbifID"].duplicated().sum())

key = ["scientificName", "decimalLatitude", "decimalLongitude", "eventDate"]
print("Vëzhgime të mundshme të dyfishta (takson + koordinata + data):", df_raw.duplicated(subset=key).sum())


3.2 DUPLIKATET
Rreshta identikë: 0
gbifID të përsëritura: 0
Vëzhgime të mundshme të dyfishta (takson + koordinata + data): 3336


## 3.3 FORMATET E eventDate

In [8]:
section("3.3 FORMATET E eventDate")
dates = df_raw["eventDate"].dropna()
fmt = (dates.str.replace(r"\d", "9", regex=True)
            .value_counts()
            .rename_axis("formati")
            .to_frame("regjistrime"))
print(fmt)

start = df_raw["eventDate"].str.split("/").str[0].str[:10]
df["eventDate"] = pd.to_datetime(start, format="ISO8601", errors="coerce")
df["eventDatePrecision"] = pd.Categorical(
    start.str.len().map({4: "year", 7: "month", 10: "day"})
)
df["isDateRange"] = df_raw["eventDate"].str.contains("/", na=False)

print("\nData të palexueshme:", (df["eventDate"].isna() & df_raw["eventDate"].notna()).sum())
print("Intervale datash:", df["isDateRange"].sum())
print(df["eventDatePrecision"].value_counts(dropna=False))


both = df["eventDate"].notna() & df["year"].notna()
mismatch = both & (df["eventDate"].dt.year != df["year"])
print("\nMospërputhje eventDate vs year:", mismatch.sum())
print("Data në të ardhmen:", (df["eventDate"] > pd.Timestamp.today()).sum())


3.3 FORMATET E eventDate
                                     regjistrime
formati                                         
9999-99-99                                 37945
9999-99-99T99:99:99                         2554
9999-99-99T99:99                            2417
9999-99-99T99:99:99.999Z                     595
9999                                         409
9999-99-99/9999-99-99                        398
9999-99                                      173
9999-99-99T99:99:99Z                          67
9999/9999                                     46
9999-99-99T99:99Z                             34
9999-99-99T99:99Z/9999-99-99T99:99Z            3

Data të palexueshme: 0
Intervale datash: 447
eventDatePrecision
day      44013
NaN       2190
year       455
month      173
Name: count, dtype: int64

Mospërputhje eventDate vs year: 0
Data në të ardhmen: 0


## 3.4 VLERAT E DYSHIMTA

In [9]:
section("3.4 VLERAT E DYSHIMTA")
print(df[["year", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters",
          "elevation", "individualCount"]].describe().T)

LAT, LON = (41.85, 43.27), (20.01, 21.80)
ELEV = (297, 2656)

checks = {
    "Koordinata jashtë Kosovës": ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON),
    "Lartësia = 0 m": df.elevation == 0,
    "Lartësia jashtë 297–2656 m (pa 0)": df.elevation.notna() & (df.elevation != 0) & ~df.elevation.between(*ELEV),
    "Pasiguria e koordinatave > 10 km": df.coordinateUncertaintyInMeters > 10_000,
    "Viti para 1900": df.year < 1900,
    "Identifikim mbi nivelin e species": df.species.isna(),
}
quality = pd.DataFrame({
    "regjistrime": {k: int(v.sum()) for k, v in checks.items()},
})
quality["perqindja"] = (quality.regjistrime / len(df) * 100).round(2)
print()
print(quality)


ic = df["individualCount"].dropna()
q1, q3 = ic.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"\nQ1={q1}, Q3={q3}, kufiri i sipërm IQR={upper}")
print("Outliers:", (ic > upper).sum(), "| Maksimumi:", ic.max())
print(ic.value_counts().head(10))


3.4 VLERAT E DYSHIMTA
                                 count         mean           std        min        25%        50%        75%        max
year                           44563.0  2015.648161     14.666617     1826.0     2018.0     2020.0     2022.0     2026.0
decimalLatitude                46831.0    42.564153      0.214627  41.858976  42.561417  42.579243  42.659067   43.24167
decimalLongitude               46831.0    20.972567      0.345911  20.020786  20.815317  21.048304  21.144554  21.762903
coordinateUncertaintyInMeters  14827.0  4865.172658  37074.786607        1.0      250.0     7071.0     7071.0  4226661.0
elevation                       9820.0   582.211845    634.895781        0.0        0.0      490.0      802.0     3050.0
individualCount                29134.0    22.948273    322.478196        1.0        1.0        2.0        6.0    10200.0

                                   regjistrime  perqindja
Koordinata jashtë Kosovës                    0       0.00
Lartësia = 0 

## 3.5 FLAMUJT E GBIF (issue)

In [10]:
section("3.5 FLAMUJT E GBIF (issue)")
issues = (df_raw["issue"].dropna()
          .str.split(";")
          .explode()
          .value_counts()
          .rename_axis("flamuri")
          .to_frame("regjistrime"))
issues["perqindja"] = (issues.regjistrime / len(df_raw) * 100).round(2)
print(issues.head(15))


3.5 FLAMUJT E GBIF (issue)
                                                  regjistrime  perqindja
flamuri                                                                 
CONTINENT_DERIVED_FROM_COORDINATES                      44725      95.50
TAXON_CONCEPT_ID_NOT_FOUND                              26628      56.86
COORDINATE_ROUNDED                                      14944      31.91
COUNTRY_DERIVED_FROM_COORDINATES                         7505      16.03
TAXON_ID_NOT_FOUND                                       6083      12.99
GEODETIC_DATUM_ASSUMED_WGS84                             3850       8.22
OCCURRENCE_STATUS_INFERRED_FROM_INDIVIDUAL_COUNT         1927       4.11
INSTITUTION_MATCH_NONE                                   1351       2.88
TAXON_MATCH_HIGHERRANK                                    756       1.61
RECORDED_DATE_INVALID                                     508       1.08
INSTITUTION_MATCH_FUZZY                                   452       0.97
COUNTRY_INVALID        

## 3.6 KONSISTENCA E VLERAVE KATEGORIKE

In [11]:
section("3.6 KONSISTENCA E VLERAVE KATEGORIKE")
for c in ["kingdom", "taxonRank", "basisOfRecord", "district", "license", "strictlyProtected", "iasUnionConcern"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")


3.6 KONSISTENCA E VLERAVE KATEGORIKE
--- kingdom ---
kingdom
Animalia          42003
Plantae            3740
Fungi              1006
Chromista            38
NaN                  37
Protozoa              4
incertae sedis        2
Bacteria              1 

--- taxonRank ---
taxonRank
SPECIES       44727
GENUS           925
SUBSPECIES      579
UNRANKED        214
FORM            132
VARIETY          60
FAMILY           58
ORDER            41
NaN              37
KINGDOM          27
PHYLUM           17
CLASS            14 

--- basisOfRecord ---
basisOfRecord
HUMAN_OBSERVATION      42479
PRESERVED_SPECIMEN      1748
OCCURRENCE              1269
MATERIAL_SAMPLE          641
MACHINE_OBSERVATION      376
MATERIAL_CITATION        317
OBSERVATION                1 

--- district ---
district
Prishtinë / Priština              25734
Prizren                            5268
Gjilan / Gnjilane                  4156
Mitrovicë / Kosovska Mitrovica     3196
Pejë / Peć                         3187
Gjakovë

## 4. RUAJTJA E RAPORTEVE

In [12]:
section("4. RUAJTJA E RAPORTEVE")
REPORTS_DIR.mkdir(exist_ok=True)
missing.to_csv(REPORTS_DIR / "01_vlerat_e_zbrazeta.csv")
quality.to_csv(REPORTS_DIR / "01_kontrollet_e_kualitetit.csv")
issues.to_csv(REPORTS_DIR / "01_gbif_issues.csv")
print("Raportet u ruajtën në", REPORTS_DIR.resolve())


4. RUAJTJA E RAPORTEVE
Raportet u ruajtën në C:\Master\PVDH\reports
